<a href="https://colab.research.google.com/github/VierickBy2008/PROYECTOS-PROPIOS-PYTHON/blob/main/EJECICIOS_DE_OPTIMIZACI%C3%93N.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#EJERCICIO 3 DE LA PPT
from itertools import combinations

def mochila_fuerza_bruta(n, W, v, w):
    mejor_valor = 0
    mejor_conjunto = ()

    # En Python, los índices de las listas empiezan en 0,
    # por lo que nuestro conjunto base será {0, 1, ..., n-1}
    elementos = range(n)

    # Generamos todos los subconjuntos posibles de todos los tamaños (de 0 a n)
    for r in range(n + 1):
        for S in combinations(elementos, r):
            peso = 0
            valor = 0

            # Calculamos el peso y valor para cada elemento en el subconjunto actual
            for i in S:
                peso += w[i]
                valor += v[i]

            # Verificamos si no excede la capacidad y si mejora el valor encontrado
            if peso <= W:
                if valor > mejor_valor:
                    mejor_valor = valor
                    mejor_conjunto = S

    return mejor_conjunto, mejor_valor

# ==========================================
# Ejemplo de uso:
# ==========================================
if __name__ == "__main__":
    # Cantidad de elementos
    n = 4
    # Capacidad máxima de la mochila
    W = 5
    # Valores de los elementos
    v = [10, 40, 30, 50]
    # Pesos de los elementos
    w = [5, 4, 6, 3]

    conjunto_optimo, valor_optimo = mochila_fuerza_bruta(n, W, v, w)

    print(f"Mejor valor encontrado: {valor_optimo}")
    # Sumamos 1 a los índices si queremos que coincida con el formato {1, 2, ..., n} del pseudocódigo
    print(f"Mejor conjunto de elementos (índices 0 a n-1): {conjunto_optimo}")

Mejor valor encontrado: 50
Mejor conjunto de elementos (índices 0 a n-1): (3,)


In [ ]:
#EJERCICIO 2 DE LA PPT
def mochila_01_pd(n, W, v, w):
    # Inicializamos una matriz V de tamaño (n+1) x (W+1) llena de ceros.
    # V[i][j] guardará el valor máximo para los primeros 'i' elementos y una capacidad 'j'.
    V = [[0 for _ in range(W + 1)] for _ in range(n + 1)]

    # Iteramos sobre los elementos (1 a n)
    for i in range(1, n + 1):
        # Iteramos sobre las capacidades de la mochila (1 a W)
        for j in range(1, W + 1):

            # NOTA SOBRE ÍNDICES:
            # En el pseudocódigo se usa w[i] y v[i] asumiendo que empiezan en 1.
            # En Python las listas empiezan en 0, por lo que el i-ésimo elemento
            # corresponde a los índices w[i-1] y v[i-1].

            if w[i-1] > j:
                # Si el peso del elemento actual es mayor que la capacidad 'j',
                # no lo podemos incluir. Tomamos el valor sin este elemento.
                V[i][j] = V[i-1][j]
            else:
                # Si lo podemos incluir, decidimos qué nos da más valor:
                # 1. No incluirlo (V[i-1][j])
                # 2. Incluirlo (sumamos su valor v[i-1] al mejor valor que teníamos para el peso restante)
                V[i][j] = max(V[i-1][j], v[i-1] + V[i-1][j - w[i-1]])

    # El resultado final se encuentra en la esquina inferior derecha de la matriz
    return V[n][W]

# ==========================================
# Ejemplo de uso:
# ==========================================
if __name__ == "__main__":
    n = 4
    W = 5
    v = [10, 40, 30, 50]
    w = [5, 4, 6, 3]

    valor_optimo = mochila_01_pd(n, W, v, w)
    print(f"Mejor valor encontrado (Programación Dinámica): {valor_optimo}")

Mejor valor encontrado (Programación Dinámica): 50


In [ ]:
#Ejercicio 1 PPT
class Objeto:
    def __init__(self, valor, peso):
        self.valor = valor
        self.peso = peso
        # Calculamos el valor por unidad de peso
        self.ratio = valor / peso

def mochila_fraccionaria(capacidad, objetos):
    # Ordenamos los objetos de mayor a menor según su ratio (valor/peso)
    objetos_ordenados = sorted(objetos, key=lambda x: x.ratio, reverse=True)
    valor_total, cap_restante = 0.0, capacidad

    for obj in objetos_ordenados:
        if cap_restante == 0:
            break

        if obj.peso <= cap_restante:
            # Si el objeto entero cabe, lo tomamos completo
            valor_total += obj.valor
            cap_restante -= obj.peso
        else:
            # Si no cabe completo, tomamos la fracción que entra en la mochila
            valor_total += obj.valor * (cap_restante / obj.peso)
            break # La mochila ya está llena y no podemos meter más

    return valor_total

# ==========================================
# Ejemplo de uso:
# ==========================================
if __name__ == "__main__":
    # Capacidad máxima de la mochila
    W = 50

    # Creamos una lista instanciando nuestra clase Objeto (valor, peso)
    lista_objetos = [
        Objeto(60, 10),  # Ratio: 6.0 (El más valioso por kg)
        Objeto(100, 20), # Ratio: 5.0
        Objeto(120, 30)  # Ratio: 4.0 (El menos valioso por kg)
    ]

    valor_optimo = mochila_fraccionaria(W, lista_objetos)

    print(f"Mejor valor encontrado (Mochila Fraccionaria): {valor_optimo}")

Mejor valor encontrado (Mochila Fraccionaria): 240.0


In [ ]:
#EJERCICIO 5 PPT
def knapsack_variant(weights, values, W):
    if len(weights) != len(values):
        raise ValueError("Dimensiones incorrectas.")

    # Emparejar pesos, valores y conservar índices originales.
    # Usamos zip para unir ambas listas y enumerate para sacar el índice 'i'.
    items = [(w, v, i) for i, (w, v) in enumerate(zip(weights, values))]

    # Ordenar de forma ascendente. Al no especificar una clave (key),
    # Python ordena por el primer elemento de la tupla, que es el peso (w).
    items.sort()

    max_value = 0
    current_weight = 0
    selected_items = []

    # Recorrido voraz (Greedy)
    for w, v, idx in items:
        # Si el peso acumulado más el peso del objeto actual no supera la capacidad
        if current_weight + w <= W:
            selected_items.append(idx)
            max_value += v
            current_weight += w
        else:
            # Condición de parada temprana: como están ordenados por peso,
            # si este no cabe, los siguientes (que son más pesados) tampoco cabrán.
            break

    return max_value, selected_items

# ==========================================
# Ejemplo de uso:
# ==========================================
if __name__ == "__main__":
    # Capacidad máxima de la mochila
    capacidad = 10

    # Lista de pesos y valores
    # Objeto 0: peso 5, valor 100 (Muy valioso, pero pesado)
    # Objeto 1: peso 2, valor 10  (Ligero)
    # Objeto 2: peso 4, valor 30  (Medio)
    # Objeto 3: peso 3, valor 20  (Ligero)
    pesos = [5, 2, 4, 3]
    valores = [100, 10, 30, 20]

    valor_total, objetos_seleccionados = knapsack_variant(pesos, valores, capacidad)

    print(f"Valor total obtenido: {valor_total}")
    print(f"Índices de los objetos seleccionados: {objetos_seleccionados}")

    # Explicación de lo que hizo el algoritmo:
    print("\n--- ¿Qué pasó aquí? ---")
    print("El algoritmo ordenó los objetos por peso: 2kg, 3kg, 4kg, 5kg.")
    print("Metió el de 2kg, luego el de 3kg, luego el de 4kg. (Total: 9kg).")
    print("Cuando intentó meter el de 5kg, ya no cabía (9 + 5 > 10) y se detuvo.")
    print("Nota que ignoró el objeto más valioso (el de 100) simplemente porque era más pesado.")

Valor total obtenido: 60
Índices de los objetos seleccionados: [1, 3, 2]

--- ¿Qué pasó aquí? ---
El algoritmo ordenó los objetos por peso: 2kg, 3kg, 4kg, 5kg.
Metió el de 2kg, luego el de 3kg, luego el de 4kg. (Total: 9kg).
Cuando intentó meter el de 5kg, ya no cabía (9 + 5 > 10) y se detuvo.
Nota que ignoró el objeto más valioso (el de 100) simplemente porque era más pesado.


In [ ]:
#Ejercicio 4 PPT
def knapsack_variant(weights, values, W):
    if len(weights) != len(values):
        raise ValueError("Dimensiones incorrectas.")

    # Emparejar pesos, valores y conservar índices originales.
    # Usamos zip para unir ambas listas y enumerate para sacar el índice 'i'.
    items = [(w, v, i) for i, (w, v) in enumerate(zip(weights, values))]

    # Ordenar de forma ascendente. Al no especificar una clave (key),
    # Python ordena por el primer elemento de la tupla, que es el peso (w).
    items.sort()

    max_value = 0
    current_weight = 0
    selected_items = []

    # Recorrido voraz (Greedy)
    for w, v, idx in items:
        # Si el peso acumulado más el peso del objeto actual no supera la capacidad
        if current_weight + w <= W:
            selected_items.append(idx)
            max_value += v
            current_weight += w
        else:
            # Condición de parada temprana: como están ordenados por peso,
            # si este no cabe, los siguientes (que son más pesados) tampoco cabrán.
            break

    return max_value, selected_items

# ==========================================
# Ejemplo de uso:
# ==========================================
if __name__ == "__main__":
    # Capacidad máxima de la mochila
    capacidad = 10

    # Lista de pesos y valores
    # Objeto 0: peso 5, valor 100 (Muy valioso, pero pesado)
    # Objeto 1: peso 2, valor 10  (Ligero)
    # Objeto 2: peso 4, valor 30  (Medio)
    # Objeto 3: peso 3, valor 20  (Ligero)
    pesos = [5, 2, 4, 3]
    valores = [100, 10, 30, 20]

    valor_total, objetos_seleccionados = knapsack_variant(pesos, valores, capacidad)

    print(f"Valor total obtenido: {valor_total}")
    print(f"Índices de los objetos seleccionados: {objetos_seleccionados}")

    # Explicación de lo que hizo el algoritmo:
    print("\n--- ¿Qué pasó aquí? ---")
    print("El algoritmo ordenó los objetos por peso: 2kg, 3kg, 4kg, 5kg.")
    print("Metió el de 2kg, luego el de 3kg, luego el de 4kg. (Total: 9kg).")
    print("Cuando intentó meter el de 5kg, ya no cabía (9 + 5 > 10) y se detuvo.")
    print("Nota que ignoró el objeto más valioso (el de 100) simplemente porque era más pesado.")

Valor total obtenido: 60
Índices de los objetos seleccionados: [1, 3, 2]

--- ¿Qué pasó aquí? ---
El algoritmo ordenó los objetos por peso: 2kg, 3kg, 4kg, 5kg.
Metió el de 2kg, luego el de 3kg, luego el de 4kg. (Total: 9kg).
Cuando intentó meter el de 5kg, ya no cabía (9 + 5 > 10) y se detuvo.
Nota que ignoró el objeto más valioso (el de 100) simplemente porque era más pesado.
